In [0]:
import os
import sys

import mlflow
from pyspark.sql import functions as F

sys.path.insert(0, os.path.abspath(".."))
from modelos import pronostico as P
from modelos import semanal as S

NOMBRE = "vigia.ml.pronostico_dengue"
TABLA = "vigia.gold.alertas"
mlflow.set_registry_uri("databricks-uc")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA} (
  dpa_provin STRING COMMENT 'Código DPA de la provincia, de 01 a 24',
  provincia STRING COMMENT 'Nombre de la provincia',
  emision DATE COMMENT 'Inicio (domingo) de la SE emitida',
  anio_epi BIGINT COMMENT 'Año epidemiológico de la SE emitida',
  semana_epi BIGINT COMMENT 'Número de la SE emitida',
  fecha_emision DATE COMMENT 'Sábado en que se publica el boletín',
  nivel STRING COMMENT 'Alerta: roja, amarilla, verde o sin canal',
  motivo STRING COMMENT 'Por qué la provincia tiene ese nivel, en una frase',
  en_epidemia BOOLEAN COMMENT 'El pronóstico a 2-4 semanas pasa el umbral epidémico (q3 del canal, mínimo 5 casos)',
  sube BOOLEAN COMMENT 'El pronóstico a 4 semanas sube lo que pide la regla del campeón',
  senal BOOLEAN COMMENT 'Zona epidémica y subida: si se repite en la emisión siguiente, la alerta es roja',
  casos_semana DOUBLE COMMENT 'Casos de la SE emitida, como se conocían al emitir',
  casos_media_4s DOUBLE COMMENT 'Promedio de casos de las 4 semanas hasta la SE emitida',
  pronostico_max DOUBLE COMMENT 'Pronóstico de la semana más alta frente a su umbral epidémico, entre 2 y 4 semanas',
  semana_max BIGINT COMMENT 'Número de la SE de pronostico_max',
  umbral_epidemia DOUBLE COMMENT 'Umbral epidémico de esa SE: el q3 del canal, con un mínimo de 5 casos',
  casos_4s DOUBLE COMMENT 'Casos pronosticados para las próximas 4 semanas',
  camas_dia_4s DOUBLE COMMENT 'Camas-día que ocuparían los casos de las próximas 4 semanas; es un orden de magnitud',
  camas_promedio DOUBLE COMMENT 'Camas ocupadas en un día cualquiera de esas 4 semanas: camas_dia_4s / 28',
  subida DOUBLE COMMENT 'Umbral de subida de la regla: 0 es la regla del plan',
  version STRING COMMENT 'Versión de vigia.ml.pronostico_dengue que pronosticó',
  calculado_en TIMESTAMP COMMENT 'Cuándo se calculó la fila'
)
COMMENT 'Alerta semanal de dengue por provincia: nivel, motivo y camas-día de las próximas 4 semanas. Una fila por provincia y emisión; cada sábado el Job agrega la emisión nueva'
""")

In [0]:
version = mlflow.MlflowClient().get_model_version_by_alias(NOMBRE, "champion")
subida = float(version.tags.get("subida", "0"))   # la regla que eligió la Fase 5
pronosticos = P.de_spark(spark.table("vigia.gold.pronosticos").toPandas())
hechas = set(P.de_spark(spark.table(TABLA).select("emision").distinct().toPandas())["emision"])
pendientes = sorted(set(pronosticos["emision"]) - hechas)
emision = dbutils.jobs.taskValues.get(taskKey="pronostico", key="emision", debugValue="corrida a mano")
print(f"Regla: subida {subida:g}. Última emisión pronosticada: {emision}. Por alertar: "
      f"{', '.join(str(e.date()) for e in pendientes) or 'nada, todo al día'}")

In [0]:
if pendientes:
    # La roja pide dos emisiones seguidas: la anterior a la primera pendiente entra solo como contexto
    contexto = pronosticos[pronosticos["emision"] >= min(pendientes) - P.SEMANA]
    nuevas = S.alertas(contexto, subida, pendientes)
    (spark.createDataFrame(P.para_spark(nuevas)).withColumn("calculado_en", F.current_timestamp())
     .write.mode("append").saveAsTable(TABLA))

ultima = spark.table(TABLA).agg(F.max("emision")).first()[0]
boletin = (spark.table(TABLA).where(F.col("emision") == ultima)
           .withColumn("orden", F.expr("CASE nivel WHEN 'roja' THEN 0 WHEN 'amarilla' THEN 1 WHEN 'verde' THEN 2 "
                                       "ELSE 3 END"))
           .orderBy("orden", F.desc("casos_4s")))
niveles = {f["nivel"]: f["count"] for f in boletin.groupBy("nivel").count().collect()}
print(f"Boletín de la emisión {ultima}: {niveles.get('roja', 0)} provincias en roja, "
      f"{niveles.get('amarilla', 0)} en amarilla y {niveles.get('verde', 0)} en verde")
display(boletin.select("provincia", "nivel", "motivo", "casos_4s", "camas_dia_4s"))